# 01 — Sabitlenmiş model yükleme doğrulaması (inference yok)

Bu notebook, Stage 3 kaynak paketindeki kanonik runner'ın `--runtime-preflight` kipini çağırır. Sabitlenmiş Qwen modelini ve tokenizer'ı yükler; model revision'ı, FP16 dtype'ını, chat template hash'ini ve çalışma ortamını kaydeder.

**Model yanıtı üretmez, vaka verisi çalıştırmaz, eğitim yapmaz ve doğrulayıcı deneyi başlatmaz.**

## Ön koşullar
- Önce `00_kaggle_environment_preflight.ipynb` çalıştırılmış olmalı.
- Kaggle GPU etkin ve internet erişimi açık olmalı.
- Stage 3 kaynak veri kümesi Input olarak eklenmiş olmalı.
- Runtime lock'taki tam Python / torch / transformers / accelerate sürümleri uyuşmuyorsa runner güvenli biçimde duracaktır. Sürüm kontrolünü atlatma.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import os
from datetime import datetime, timezone

manifest_hits = list(Path("/kaggle/input").rglob("EXPERIMENT_MANIFEST_STAGE3.json"))
if len(manifest_hits) != 1:
    raise RuntimeError(
        f"Tam olarak 1 Stage 3 manifest bekleniyordu; bulunan: {len(manifest_hits)}. "
        "Önce Kaggle Input ayarlarını düzelt."
    )
repo_root = manifest_hits[0].parents[2]
runner = repo_root / "research_candidates" / "evidence_abstention" / "kaggle_stage3_p0_p1.py"
if not runner.is_file():
    raise FileNotFoundError(f"Kanonik runner bulunamadı: {runner}")

print("Kaynak kökü:", repo_root)
print("Runner:", runner)
print("Bu çalıştırma model yanıtı üretmeyecek.")


In [ ]:
# Önceki raporu asla ezme; her çalıştırmada yeni bir rapor yolu seç.
working = Path("/kaggle/working")
working.mkdir(parents=True, exist_ok=True)
base = "runtime_preflight_report"
candidate = working / f"{base}.json"
index = 1
while candidate.exists():
    candidate = working / f"{base}_{index:02d}.json"
    index += 1

env = os.environ.copy()
env["EVIDENCE_RUNTIME_REPORT_PATH"] = str(candidate)
cmd = [sys.executable, str(runner), "--root", str(repo_root), "--runtime-preflight"]
print("Komut:", " ".join(cmd))
print("Rapor yolu:", candidate)

proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
print("Exit code:", proc.returncode)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print("STDERR:")
    print(proc.stderr)

if proc.returncode != 0:
    print("\nMODEL YÜKLEME DOĞRULAMASI GEÇMEDİ.")
    print("Bu durumda sürüm/GPU/model erişim hatasını kaydet; doğrulama kapılarını değiştirme.")
else:
    if not candidate.is_file():
        raise RuntimeError("Runner başarılı döndü ama runtime rapor dosyası oluşmadı.")
    report = json.loads(candidate.read_text(encoding="utf-8"))
    if report.get("inference_performed") is not False or report.get("generated_tokens") != 0:
        raise RuntimeError("Beklenmeyen durum: preflight raporu inference yapılmadığını doğrulamıyor.")
    print("\nMODEL YÜKLEME DOĞRULAMASI GEÇTİ; BU, ANA DENEY İZNİ DEĞİLDİR.")
    print("Model:", report.get("model_id"))
    print("Revision:", report.get("resolved_model_revision"))
    print("Tokenizer revision:", report.get("resolved_tokenizer_revision"))
    print("GPU:", report.get("gpu_name"))
    print("Runtime fingerprint:", report.get("runtime_fingerprint_sha256"))


## Çıktıların anlamı

- Başarılı yükleme yalnızca **model/tokenizer/runtime teknik olarak yüklenebildi** demektir.
- Bu, araştırma veri kümesinin kabul edildiği, ön kaydın tamamlandığı veya inference izni verildiği anlamına gelmez.
- Bu notebookta herhangi bir model yanıtı üretilmemelidir.
- `EVIDENCE_ABSTENTION_INFERENCE_APPROVED` değişkenini ayarlama; bu notebook onu kullanmaz.